In [ ]:
# read:  ner_results_raw.parquet
# write: company_industry_lookup.csv, ner_results.parquet

import pandas as pd
import json
import time
from tqdm import tqdm
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/nlp_final_project/'

MessageError: Error: credential propagation was unsuccessful

In [ ]:
!pip install -q -U google-genai

In [ ]:
from google import genai
from google.colab import userdata

api_key = userdata.get('GEMINI_API_KEY')
model = 'gemini-3-flash'
client = genai.Client(api_key=api_key)

In [ ]:
# for model in client.models.list():
#     print(model.name)

In [ ]:
# # API test
# response = client.models.generate_content(
#     model='gemini-2.5-flash-lite',
#     contents='Explain how an API works in a short single sentence.'
# )

# print(response.text)

In [ ]:
org_df = pd.read_parquet(path + 'ner_orgs_raw.parquet')

org_df_exploded = org_df.explode('org').reset_index(drop=True)
org_df_exploded = org_df_exploded.dropna(subset=['org'])
org_df_exploded = org_df_exploded[org_df_exploded['org'].str.strip() != '']

print(f"Total Organizations: {len(org_df_exploded)}")
print(f"Unique articles With Orgs: {org_df_exploded['article_id'].nunique()}")

In [ ]:
org_counts_per_article = org_df_exploded.groupby('article_id')['org'].count()
org_counts_per_article.describe()

import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

org_counts_per_article.hist(bins=50, ax=axes[0])
axes[0].set_title('ORG Count Distribution Per Article')
axes[0].set_xlabel('Number of ORGs')
axes[0].set_ylabel('Number of Articles')

# less than 20
org_counts_per_article[org_counts_per_article <= 20].hist(bins=50, ax=axes[1])
axes[1].set_title('ORG Count Distribution (20/less Orgs)')
axes[1].set_xlabel('Number of ORGs')
axes[1].set_ylabel('Number of Articles')

plt.tight_layout()
plt.show()

In [ ]:
# percentiles
for p in [10, 25, 50, 75, 90, 95, 99]:
    print(f"{p}th percentile: {org_counts_per_article.quantile(p/100):.0f} orgs")

In [ ]:
MAX_ORGS_PER_ARTICLE = 10

articles_before = org_df_exploded['article_id'].nunique()

valid_article_ids = org_counts_per_article[
    org_counts_per_article <= MAX_ORGS_PER_ARTICLE
].index

org_df_exploded = org_df_exploded[
    org_df_exploded['article_id'].isin(valid_article_ids)
].copy()

articles_after = org_df_exploded['article_id'].nunique()
print(f"Articles before threshold filter: {articles_before}")
print(f"Articles after threshold filter:  {articles_after}")
print(f"Articles dropped: {articles_before - articles_after}")

In [ ]:
company_counts = (
    org_df_exploded['org']
    .value_counts()
    .reset_index()
    .rename(columns={'org': 'company', 'count': 'mention_count'})
)

print(f"Total unique companies: {len(company_counts)}")

In [ ]:
company_counts['mention_count'].describe()

In [ ]:
company_counts.head()

In [ ]:
company_counts.tail()

In [ ]:
once = (company_counts['mention_count'] <= 5).sum()
print(f"Companies mentioned 5 times or less: {once} ({once/len(company_counts)*100:.1f}%)")

In [ ]:
company_counts.hist(bins=50, figsize=(10, 4),log =True)
plt.title('Company Mention Frequency Distribution')
plt.xlabel('Mention Count')
plt.ylabel('Number of Companies')
plt.show()

In [ ]:
MIN_MENTIONS = 5

companies_to_classify = company_counts[
    company_counts['mention_count'] >= MIN_MENTIONS
]['company'].tolist()

print(f"Companies to classify (mentioned {MIN_MENTIONS}+ times): {len(companies_to_classify)}")
print(f"API calls needed at 50 per batch: {len(companies_to_classify) // 50 + 1}")

I used an API call to gemini to assign industries to the companies, to guide the model, I provided a preset list of industries from which it could pick an industry to attach as well as an unknown option incase the model wasn't able to identify an industry.

In [ ]:
from tqdm import tqdm
from google.genai import types
import json
import time

INDUSTRIES = [
    "Technology", "Healthcare", "Finance", "Legal",
    "Manufacturing", "Retail", "Energy", "Media",
    "Transportation", "Education", "Government",
    "Real Estate", "Telecommunications", "Unknown"
]

def classify_companies_batch(company_batch):
    companies_str = "\n".join(company_batch)
    prompt = f"""
    Classify each company below into strictly one industry from this list:
    {', '.join(INDUSTRIES)}

    Return ONLY a valid JSON object mapping each company name exactly
    as given to its industry. Example format: {{"Company A": "Technology", "Company B": "Finance"}}

    Companies:
    {companies_str}
    """

    max_retries = 2
    for attempt in range(max_retries):
      try:
          response = client.models.generate_content(
              model ='gemini-3.1-flash-lite-preview',
              contents = prompt,
              config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.0,
                )
              )

          return json.loads(response.text)

      except Exception as e:
          print(f"Batch failed on attempt {attempt +1}: {e}")
          return {company: "Unknown" for company in company_batch}
          time.sleep(5)


In [ ]:
import os
# make api call
BATCH_SIZE = 50
industry_lookup = {}
batches = [
    companies_to_classify[i:i + BATCH_SIZE]
    for i in range(0, len(companies_to_classify), BATCH_SIZE)
]
for i, batch in enumerate(tqdm(batches)):
    result = classify_companies_batch(batch)
    industry_lookup.update(result)

    if (i+1) % 10 == 0:
        checkpoint_df = pd.DataFrame(
            list(industry_lookup.items()),
            columns=['company', 'industry']
        )
        checkpoint_df.to_csv(path + f'industry_lookup_checkpoint_{i}.csv', index=False)
    time.sleep(5)

The API call timed out due to limited RPM on the "gemini-2.5-flash" model. I changed the model to "gemini-3.1-flash-preview" that has a higher limit and can handle such rigid tasks such as classification based on a preset list of industries.

In [ ]:
company_industry_lookup = pd.read_csv(path + 'industry_lookup_checkpoint_409.csv')

company_industry_lookup = company_industry_lookup.merge(
    company_counts,
    on='company',
    how='left'
)

company_industry_lookup.to_csv(path + 'company_industry_lookup.csv', index=False)
print(f"Lookup table: {len(company_industry_lookup)} companies classified")
print(company_industry_lookup['industry'].value_counts())
print(company_industry_lookup.head())

industry_map = company_industry_lookup.set_index('company')['industry'].to_dict()
org_df_final = org_df_exploded.copy()
org_df_final['industry'] = org_df_final['org'].map(industry_map).fillna('Unknown')

print(f"Final NER results shape: {org_df_final.shape}")
print(org_df_final['industry'].value_counts())
print(org_df_final.head())

org_df_final.to_parquet(path + 'ner_results.parquet', index=False)